In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit, col, to_timestamp
from pyspark.sql.types import *

source_path = "/Volumes/iotmlhealthcatalog/default/iotbatch/vitaldb_data"

schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("caseid", IntegerType(), True),
    StructField("SNUADC_ART_SBP", DoubleType(), True),
    StructField("Solar8000_HR", DoubleType(), True),
    StructField("Solar8000_PLETH_SPO2", DoubleType(), True),
    StructField("Solar8000_BT", DoubleType(), True),
    StructField("Device_Battery_Level", DoubleType(), True),
    StructField("Operator_ID", IntegerType(), True),
    StructField("target", IntegerType(), True)
])

# =========================
# LECTURE
# =========================
raw_df = (
    spark.read.format("csv")
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .schema(schema)
        .load(source_path)
)

raw_df.printSchema()

# =========================
# FIX TIMESTAMP EXPLICITLY
# =========================
raw_df = raw_df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"))
)

# =========================
# BRONZE
# =========================
bronze_df = (
    raw_df
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("source_type", lit("batch"))
)

# =========================
# WRITE
# =========================
bronze_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("iotmlhealthcatalog.bronze.vitaldbtrain")

print("✅ Bronze OK")